# RAG Pipeline — Demo complet
Chargement → Découpage → Embeddings → Index → Recherche sémantique

## 1. Configuration

In [ ]:
import sys, os
# Make sure the project root is on the path
sys.path.insert(0, os.path.abspath("..") if os.path.basename(os.getcwd()) != "projet" else ".")

# ── Paramètres à modifier selon votre besoin ──────────────────────────────────
DOCUMENTS_FOLDER = "data/enzymes"        # dossier à indexer
INDEX_SAVE_PATH  = "data/enzymes_index.pkl"  # où persister l'index
TOP_K            = 3                     # nombre de résultats souhaités
QUERY = "Améliorant de panification : quelles sont les quantités recommandées d'alpha-amylase, xylanase et d'Acide ascorbique ?"

print("✅ Configuration chargée.")
print(f"   Dossier  : {DOCUMENTS_FOLDER}")
print(f"   Index    : {INDEX_SAVE_PATH}")
print(f"   Top-K    : {TOP_K}")

: 

## 2. Chargement des documents et découpage en fragments

In [ ]:
from rag_module.loader import DocumentLoader

loader = DocumentLoader(chunk_size=512, chunk_overlap=64)
fragments = loader.load_directory(DOCUMENTS_FOLDER)

print(f"📄 {len(fragments)} fragments chargés depuis '{DOCUMENTS_FOLDER}'")
print(f"\n── Aperçu des 3 premiers fragments ──")
for f in fragments[:3]:
    print(f"\n[Source: {f.source} | Page: {f.page}]")
    print(f.text[:250].strip(), "…")

## 3. Génération des embeddings

In [ ]:
from rag_module.embeddings import EmbeddingModel

model = EmbeddingModel()  # paraphrase-multilingual-MiniLM-L12-v2

texts = [f.text for f in fragments]
embeddings = model.encode_documents(texts, show_progress=True)

print(f"\n✅ Embeddings générés")
print(f"   Forme de la matrice : {embeddings.shape}  (fragments × dimensions)")
print(f"   dtype : {embeddings.dtype}")

## 4. Construction et sauvegarde de l'index vectoriel

In [ ]:
from rag_module.indexer import VectorIndex

index = VectorIndex()
index.add(fragments, embeddings)

index.save(INDEX_SAVE_PATH)

print(f"✅ Index construit et sauvegardé dans '{INDEX_SAVE_PATH}'")
print(f"   Fragments indexés : {index.size}")
print(f"   Dimension         : {index.embedding_dim}")

## 5. Recherche sémantique

In [ ]:
from rag_module.searcher import SemanticSearcher

searcher = SemanticSearcher(model, index, top_k=TOP_K)

print(f"🔍 Question : {QUERY}\n")
results = searcher.search(QUERY, top_k=TOP_K)
searcher.display_results(QUERY, results)

## 6. Visualisation des scores

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Résultat":       [f"#{r.rank} — {r.fragment.source}" for r in results],
    "Score cosinus":  [round(r.score, 4) for r in results],
    "Similarité (%)": [round(r.score_percent, 1) for r in results],
    "Extrait":        [r.fragment.text[:120].strip() + "…" for r in results],
})

print(df.to_string(index=False))

# Bar chart (works in any notebook renderer)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 3))
    colors = ["#2ecc71" if s >= 70 else "#f39c12" if s >= 50 else "#e74c3c"
              for s in df["Similarité (%)"]]
    ax.barh(df["Résultat"][::-1], df["Similarité (%)"][::-1], color=colors[::-1])
    ax.set_xlabel("Similarité cosinus (%)")
    ax.set_title(f"Top-{TOP_K} résultats")
    ax.set_xlim(0, 100)
    for i, (val, label) in enumerate(zip(df["Similarité (%)"][::-1], df["Résultat"][::-1])):
        ax.text(val + 0.5, i, f"{val}%", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("(matplotlib non installé — graphique ignoré)")

---
## Bonus — Version raccourcie avec `RAGPipeline`
Les 6 étapes ci-dessus correspondent exactement à ce que fait `RAGPipeline` en interne. Voici le même résultat en 4 lignes :

In [ ]:
from rag_module.pipeline import RAGPipeline

pipeline = RAGPipeline(top_k=TOP_K)
pipeline.index_directory(DOCUMENTS_FOLDER)
results2 = pipeline.search(QUERY)
pipeline.display(QUERY, results2)